<a href="https://colab.research.google.com/github/urvashi5555/Fake-news-detection-comparing-classical-ML-neural-networks-and-transformers/blob/main/02_CNN_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.utils import to_categorical

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

fake["label"] = 0
real["label"] = 1

df = pd.concat([fake, real], ignore_index=True)

df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)

(44898, 2)


In [ ]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+|www\S+|https\S+", '', text)

    text = re.sub(r'<.*?>', '', text)

    text = re.sub(r'\d+', '', text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

df["content"] = df["content"].apply(clean_text)

print("Cleaning completed")

Cleaning completed


In [ ]:
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 35918
Test size: 8980


In [ ]:
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE)

# learn vocabulary only from training data
tokenizer.fit_on_texts(X_train)

# convert text to integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test)

print("Tokenization completed")

Tokenization completed


In [ ]:
MAX_LEN = 150

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN
)

print(X_train_pad.shape)

print(X_test_pad.shape)

(35918, 150)
(8980, 150)


In [ ]:
y_train_cat = to_categorical(
    y_train,
    num_classes=2
)

y_test_cat = to_categorical(
    y_test,
    num_classes=2
)

print(y_train_cat.shape)

print(y_test_cat.shape)

(35918, 2)
(8980, 2)


In [ ]:
cnn_model = Sequential()

# EMBEDDING LAYER
cnn_model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=32,
        input_length=MAX_LEN
    )
)

# CONVOLUTION LAYER
cnn_model.add(
    Conv1D(
        filters=128,
        kernel_size=5,
        activation='relu'
    )
)

# POOLING
cnn_model.add(GlobalMaxPooling1D())

# DENSE LAYER
cnn_model.add(Dense(64, activation='relu'))

cnn_model.add(Dropout(0.3))

# OUTPUT
cnn_model.add(Dense(2, activation='softmax'))

# COMPILE
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# SUMMARY
cnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_cnn = cnn_model.fit(
    X_train_pad,
    y_train_cat,
    epochs=3,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 24s 44ms/step - accuracy: 0.9345 - loss: 0.1475 - val_accuracy: 0.9894 - val_loss: 0.0325
Epoch 2/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 23s 45ms/step - accuracy: 0.9951 - loss: 0.0175 - val_accuracy: 0.9919 - val_loss: 0.0243
Epoch 3/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 41s 44ms/step - accuracy: 0.9991 - loss: 0.0038 - val_accuracy: 0.9911 - val_loss: 0.0276


In [ ]:
cnn_pred = cnn_model.predict(X_test_pad)

cnn_pred = np.argmax(cnn_pred, axis=1)

print("Prediction completed")

281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Prediction completed


In [ ]:
cnn_accuracy = accuracy_score(y_test, cnn_pred)

print("CNN Test Accuracy:", cnn_accuracy)

print("CNN Test Accuracy (%):", cnn_accuracy * 100)

CNN Test Accuracy: 0.9899777282850779
CNN Test Accuracy (%): 98.99777282850779


In [ ]:
cnn_model = Sequential()

# EMBEDDING
cnn_model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    )
)

# CONVOLUTION 1
cnn_model.add(
    Conv1D(
        filters=128,
        kernel_size=5,
        activation='relu'
    )
)

# CONVOLUTION 2
cnn_model.add(
    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu'
    )
)

# POOLING
cnn_model.add(GlobalMaxPooling1D())

# DENSE
cnn_model.add(Dense(128, activation='relu'))

cnn_model.add(Dropout(0.5))

# OUTPUT
cnn_model.add(Dense(2, activation='softmax'))

# COMPILE
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

history_cnn = cnn_model.fit(
    X_train_pad,
    y_train_cat,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 89s 170ms/step - accuracy: 0.9340 - loss: 0.1412 - val_accuracy: 0.9919 - val_loss: 0.0279
Epoch 2/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 143s 173ms/step - accuracy: 0.9945 - loss: 0.0173 - val_accuracy: 0.9908 - val_loss: 0.0276
Epoch 3/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 85s 169ms/step - accuracy: 0.9992 - loss: 0.0029 - val_accuracy: 0.9908 - val_loss: 0.0414
Epoch 4/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 86s 170ms/step - accuracy: 0.9991 - loss: 0.0029 - val_accuracy: 0.9891 - val_loss: 0.0422
Epoch 5/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 143s 172ms/step - accuracy: 0.9996 - loss: 0.0016 - val_accuracy: 0.9897 - val_loss: 0.0409


In [ ]:
cnn_pred = cnn_model.predict(X_test_pad)

cnn_pred = np.argmax(cnn_pred, axis=1)

print("Predictions completed")

281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step
Predictions completed


In [ ]:
cnn_accuracy = accuracy_score(y_test, cnn_pred)

print("CNN Test Accuracy:", cnn_accuracy)

print("CNN Test Accuracy (%):", cnn_accuracy * 100)

CNN Test Accuracy: 0.9889755011135858
CNN Test Accuracy (%): 98.89755011135858
